# Dialog Summarizer with Transformer

In [81]:
import numpy as np
import pandas as pd
import tensorflow as tf
import time
import utils

import textwrap
wrapper = textwrap.TextWrapper(width=70)

tf.keras.utils.set_random_seed(10)
import warnings
warnings.filterwarnings('ignore')

In [82]:
tf.config.run_functions_eagerly(True)

## 1. Dataset

In [83]:
data_dir = "data/corpus"
train_data, test_data = utils.get_train_test_data(data_dir) # len(train_data)=14732, len(test_data)=819
train_data

,summary,dialogue
0,Amanda baked cookies and will bring Jerry some...,Amanda: I baked cookies. Do you want some?\r\...
1,Olivia and Olivier are voting for liberals in ...,Olivia: Who are you voting for in this electio...
2,Kim may try the pomodoro technique recommended...,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa..."
3,Edward thinks he is in love with Bella. Rachel...,"Edward: Rachel, I think I'm in ove with Bella...."
4,"Sam is confused, because he overheard Rick com...",Sam: hey overheard rick say something\r\nSam:...
...,...,...
14727,Romeo is trying to get Greta to add him to her...,Romeo: You are on my ‘People you may know’ lis...
14728,Theresa is at work. She gets free food and fre...,Theresa: <file_photo>\r\nTheresa: <file_photo>...
14729,Japan is going to hunt whales again. Island an...,John: Every day some bad news. Japan will hunt...
14730,Celia couldn't make it to the afternoon with t...,Jennifer: Dear Celia! How are you doing?\r\nJe...


### Preprocess train_data

In [84]:
document, summary = utils.preprocess(train_data)
document_test, summary_test = utils.preprocess(test_data)

# The [ and ] from default tokens cannot be removed, because they mark the SOS and EOS token.
filters = '!"#$%&()*+,-./:;<=>?@\\^_`{|}~\t\n'
oov_token = '[UNK]'

tokenizer = tf.keras.preprocessing.text.Tokenizer(filters=filters, oov_token=oov_token, lower=False)

documents_and_summary = pd.concat([document, summary], ignore_index=True)

tokenizer.fit_on_texts(documents_and_summary)

inputs = tokenizer.texts_to_sequences(document)
targets = tokenizer.texts_to_sequences(summary)

print('vocab:', tokenizer.word_index)
vocab_size = len(tokenizer.word_index) + 1 # 34250
print(f'Size of vocabulary: {vocab_size}')

# Limit the size of the input and output data for being able to run it in this environment.
encoder_maxlen = 150
decoder_maxlen = 50

# Pad the sequences.
inputs = tf.keras.preprocessing.sequence.pad_sequences(inputs, maxlen=encoder_maxlen, padding='post', truncating='post')
targets = tf.keras.preprocessing.sequence.pad_sequences(targets, maxlen=decoder_maxlen, padding='post', truncating='post')

inputs = tf.cast(inputs, dtype=tf.int32) # shape=(14732, 150), inputs[0]=[7    8  456  2 ... 0  0], len=150
targets = tf.cast(targets, dtype=tf.int32) # shape=(14732, 50), targets[0]= 7    8  456  3502 ... 0  0], len=50
# 7 và 8 là id của SOS và sos

# Create the final training dataset.
BUFFER_SIZE = 10000
BATCH_SIZE = 64

dataset = tf.data.Dataset.from_tensor_slices((inputs, targets)).shuffle(BUFFER_SIZE).batch(BATCH_SIZE) # len=231

vocab: {'[UNK]': 1, 'i': 2, 'the': 3, 'to': 4, 'you': 5, 'a': 6, '[SOS]': 7, '[EOS]': 8, 'and': 9, 'it': 10, 'is': 11, 'for': 12, 'in': 13, 'of': 14, 'will': 15, 'that': 16, 'have': 17, 'but': 18, 'so': 19, 'are': 20, 'be': 21, 'on': 22, 'at': 23, 'me': 24, 'with': 25, 'what': 26, 'not': 27, 'we': 28, 'my': 29, 'do': 30, "i'm": 31, 'know': 32, 'was': 33, "it's": 34, 'about': 35, 'just': 36, 'he': 37, 'this': 38, 'can': 39, 'no': 40, 'her': 41, 'ok': 42, 'she': 43, 'like': 44, 'they': 45, "don't": 46, 'there': 47, 'your': 48, 'go': 49, 'some': 50, 'good': 51, 'going': 52, 'see': 53, 'how': 54, 'if': 55, 'up': 56, 'time': 57, 'think': 58, 'as': 59, 'one': 60, 'get': 61, 'yeah': 62, "i'll": 63, 'all': 64, 'yes': 65, 'sure': 66, 'from': 67, 'file': 68, 'too': 69, 'has': 70, 'really': 71, 'well': 72, 'now': 73, 'him': 74, 'out': 75, 'his': 76, 'come': 77, 'when': 78, 'did': 79, 'need': 80, 'want': 81, 'd': 82, 'would': 83, 'or': 84, 'then': 85, 'thanks': 86, 'oh': 87, 'them': 88, 'an': 89, 

## 2. Model
### 2.1. Positional Encoding
$$
PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)
$$
$$
PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)
$$

In [85]:
def positional_encoding(positions, d_model): # positions=256, d_model=128
    position = np.arange(positions)[:, np.newaxis] # shape=(256, 1), =[[0] [1] ... [255]]

    k = np.arange(d_model)[np.newaxis, :] # shape=(1, 128), =[[0  1  ... 127]]
    i = k // 2 # shape=(1, 128), =[[0  0 1  1  2  2  ... 62  62  63  63]]
    
    # initialize a matrix angle_rads of all the angles 
    angle_rates = 1 / np.power(10000, (2 * i) / np.float32(d_model)) # shape=(1, 128), =[[1  1  0.866  0.866  ...  0.00015  0.00015]]
    angle_rads = position * angle_rates # shape=(256, 128)
    # [[0  0  0  ...  0]
    #  [1  1  0.866  0.866 ... 0.00015  0.00015]
    #  ...
    #  [255  255  220.83  220.83 ... 0.038  0.038]]
  
    # apply sin to even indices in the array; 2i
    angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
  
    # apply cos to odd indices in the array; 2i+1
    angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])
    
    pos_encoding = angle_rads[np.newaxis, ...] # shape=(1, 256, 128)
    
    return tf.cast(pos_encoding, dtype=tf.float32)

### 2.2. Masking

In [86]:
def create_padding_mask(decoder_token_ids):
    seq = 1 - tf.cast(tf.math.equal(decoder_token_ids, 0), tf.float32)
    # add extra dimensions to add the padding to the attention logits. 
    # this will allow for broadcasting later when comparing sequences
    return seq[:, tf.newaxis, :] 


def create_look_ahead_mask(sequence_length):
    mask = tf.linalg.band_part(tf.ones((1, sequence_length, sequence_length)), -1, 0)
    
    return mask 

### 2.3. Self-Attention

In [87]:
def scaled_dot_product_attention(q, k, v, mask):
    # Multiply q and k transposed.
    matmul_qk = tf.matmul(q, k, transpose_b=True)

    # scale matmul_qk with the square root of dk
    dk = tf.cast(k.shape[1], tf.float32)
    scaled_attention_logits = matmul_qk/tf.sqrt(dk)

    # add the mask to the scaled tensor.
    if mask is not None:  # Don't replace this None
        scaled_attention_logits += (1.0 - mask)*-1e9

    # softmax is normalized on the last axis (seq_len_k) so that the scores add up to 1.
    attention_weights = tf.keras.activations.softmax(scaled_attention_logits, axis=-1)

    # Multiply the attention weights by v
    output = tf.matmul(attention_weights, v)

    return output, attention_weights

### 2.4. Encoder

<img src="images/encoder_layer.png" height="400"/>
<img src="images/encoder.png" height="400"/>
<img src="images/self-attention.png" height="400"/>

$$
\text { Attention }(Q, K, V)=\operatorname{softmax}\left(\frac{Q K^{T}}{\sqrt{d_{k}}}+{M}\right) V\tag{4}\
$$

* $Q$ is the matrix of queries 
* $K$ is the matrix of keys
* $V$ is the matrix of values
* $M$ is the optional mask you choose to apply 
* ${d_k}$ is the dimension of the keys, which is used to scale everything down so the softmax doesn't explode


In [88]:
def FullyConnected(embedding_dim, fully_connected_dim):
    return tf.keras.Sequential([
        tf.keras.layers.Dense(fully_connected_dim, activation='relu'),  # (batch_size, seq_len, d_model)
        tf.keras.layers.Dense(embedding_dim)  # (batch_size, seq_len, d_model)
    ])


class EncoderLayer(tf.keras.layers.Layer):
    def __init__(self, embedding_dim, num_heads, fully_connected_dim,
                 dropout_rate=0.1, layernorm_eps=1e-6):
        
        super(EncoderLayer, self).__init__()

        self.mha = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads, # num_heads=2
            key_dim=embedding_dim,
            dropout=dropout_rate
        )

        self.ffn = FullyConnected(
            embedding_dim=embedding_dim,
            fully_connected_dim=fully_connected_dim
        )

        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=layernorm_eps)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=layernorm_eps)

        self.dropout_ffn = tf.keras.layers.Dropout(dropout_rate)
    
    def call(self, x, training, mask):
        # x.shape=(150, 128), mask.shape=(1, 150)
        # calculate self-attention using mha(~1 line).
        # Dropout is added by Keras automatically if the dropout parameter is non-zero during training
        self_mha_output = self.mha(query=x, value=x, key=x, attention_mask=mask)  # shape=(150, 128)
        
        # Cộng x với self_mha_output
        # skip connection
        # apply layer normalization on sum of the input and the attention output to get the  
        # output of the multi-head attention layer
        skip_x_attention = self.layernorm1(x + self_mha_output)  # (batch_size, input_seq_len, fully_connected_dim)

        # Đi qua 2 lớp FC, lớp cuối có units=embedding_dim=128
        # pass the output of the multi-head attention layer through a ffn
        ffn_output = self.ffn(skip_x_attention)  # shape=(150, 128)
        
        # apply dropout layer to ffn output during training
        # use `training=training`
        ffn_output = self.dropout_ffn(ffn_output, training=training)
        
        # Cộng skip_x_attention với ffn_output
        # apply layer normalization on sum of the output from multi-head attention (skip connection) and ffn output
        # to get the output of the encoder layer
        encoder_layer_out = self.layernorm2(skip_x_attention + ffn_output)  # shape=(150, 128)
        
        return encoder_layer_out


class Encoder(tf.keras.layers.Layer):
    def __init__(self, num_layers, embedding_dim, num_heads, fully_connected_dim, input_vocab_size,
               maximum_position_encoding, dropout_rate=0.1, layernorm_eps=1e-6):
        super(Encoder, self).__init__()

        self.embedding_dim = embedding_dim
        self.num_layers = num_layers

        self.embedding = tf.keras.layers.Embedding(input_vocab_size, self.embedding_dim)
        self.pos_encoding = positional_encoding(maximum_position_encoding, 
                                                self.embedding_dim)


        self.enc_layers = [EncoderLayer(embedding_dim=self.embedding_dim,
                                        num_heads=num_heads,
                                        fully_connected_dim=fully_connected_dim,
                                        dropout_rate=dropout_rate,
                                        layernorm_eps=layernorm_eps) 
                           for _ in range(self.num_layers)]

        self.dropout = tf.keras.layers.Dropout(dropout_rate)
        
    def call(self, x, training, mask):
        # x dài 150, mask.shape=(1, 150)

        seq_len = tf.shape(x)[1] # 150
        
        # Lấy embedding từ x
        # Pass input through the Embedding layer
        x = self.embedding(x)  # shape=(150, 128)

        # Nhân x với sqrt(embedding_dim)
        # Scale embedding by multiplying it by the square root of the embedding dimension
        x *= tf.math.sqrt(tf.cast(self.embedding_dim, tf.float32)) # shape=(150, 128)

        # Cộng x với pos_encoding; pos_encoding có shape (256, 128). Xem ở positional_encoding
        # Add the position encoding to embedding
        x += self.pos_encoding[:, :seq_len, :] # shape=(150, 128)

        # Pass the encoded embedding through a dropout layer
        # use `training=training`
        x = self.dropout(inputs=x, training=training) # shape=(150, 128)

        # Pass the output through the stack of encoding layers 
        for i in range(self.num_layers): # num_layers=2, xem tiếp ở EncoderLayer
            x = self.enc_layers[i](x=x, training=training, mask=mask)
        # x.shape=(150, 128)

        return x 

### 2.5. Decoder

<img src="images/decoder_layer.png" height="400"/>
<img src="images/decoder.png" height="400"/>

In [89]:
class DecoderLayer(tf.keras.layers.Layer):
    def __init__(self, embedding_dim, num_heads, fully_connected_dim, dropout_rate=0.1, layernorm_eps=1e-6):
        super(DecoderLayer, self).__init__()

        self.mha1 = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embedding_dim,
            dropout=dropout_rate
        )

        self.mha2 = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embedding_dim,
            dropout=dropout_rate
        )

        self.ffn = FullyConnected(
            embedding_dim=embedding_dim,
            fully_connected_dim=fully_connected_dim
        )

        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=layernorm_eps)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=layernorm_eps)
        self.layernorm3 = tf.keras.layers.LayerNormalization(epsilon=layernorm_eps)

        self.dropout_ffn = tf.keras.layers.Dropout(dropout_rate)
    
    def call(self, x, enc_output, training, look_ahead_mask, padding_mask):
        # x.shape=(49, 128), enc_output.shape=(150, 128), look_ahead_mask.shape=(49, 49), dec_padding_mask.shape=(1, 150)
        # MHA1 là self-attention có QKV là target, MHA2 là attention có Q là output từ MHA1 (sau khi cộng với target), KV là enc_output

        # BLOCK 1
        # calculate self-attention and return attention scores as attn_weights_block1.
        # Dropout will be applied during training (~1 line).
        mult_attn_out1, attn_weights_block1 = self.mha1(query=x,
                                                       value=x,
                                                       key=x,
                                                       attention_mask=look_ahead_mask,
                                                       return_attention_scores=True,
                                                       training=training)
        # mult_attn_out1.shape=(49, 128)
        
        # Cộng x với mult_attn_out1
        # apply layer normalization (layernorm1) to the sum of the attention output and the input (~1 line)
        Q1 = self.layernorm1(x + mult_attn_out1)

        # BLOCK 2
        # calculate self-attention using the Q from the first block and K and V from the encoder output. 
        # Dropout will be applied during training
        # Return attention scores as attn_weights_block2 (~1 line) 
        mult_attn_out2, attn_weights_block2 = self.mha2(query=Q1,
                                                       value=enc_output,
                                                       key=enc_output,
                                                       attention_mask=padding_mask,
                                                       return_attention_scores=True,
                                                       training=training)
        # mult_attn_out2.shape=(49, 128)
        
        # Cộng Q1 với mult_attn_out2
        # # apply layer normalization (layernorm2) to the sum of the attention output and the Q from the first block (~1 line)
        mult_attn_out2 = self.layernorm1(Q1 + mult_attn_out2)
        
        # BLOCK 3
        # Đi qua 2 lớp FC, lớp cuối có units=embedding_dim=128
        # pass the output of the second block through a ffn
        ffn_output = self.ffn(mult_attn_out2)
        
        # apply a dropout layer to the ffn output
        # use `training=training`
        ffn_output = self.dropout_ffn(inputs=ffn_output, training=training)
        
        # Cộng mult_attn_out2 với ffn_output
        # apply layer normalization (layernorm3) to the sum of the ffn output and the output of the second block
        out3 = self.layernorm3(mult_attn_out2 + ffn_output) # shape=(49, 128)

        return out3, attn_weights_block1, attn_weights_block2


class Decoder(tf.keras.layers.Layer):
    def __init__(self, num_layers, embedding_dim, num_heads, fully_connected_dim, target_vocab_size,
               maximum_position_encoding, dropout_rate=0.1, layernorm_eps=1e-6):
        super(Decoder, self).__init__()

        self.embedding_dim = embedding_dim
        self.num_layers = num_layers

        self.embedding = tf.keras.layers.Embedding(target_vocab_size, self.embedding_dim)
        self.pos_encoding = positional_encoding(maximum_position_encoding, self.embedding_dim)

        self.dec_layers = [DecoderLayer(embedding_dim=self.embedding_dim,
                                        num_heads=num_heads,
                                        fully_connected_dim=fully_connected_dim,
                                        dropout_rate=dropout_rate,
                                        layernorm_eps=layernorm_eps) 
                           for _ in range(self.num_layers)]
        self.dropout = tf.keras.layers.Dropout(dropout_rate)
    
    def call(self, x, enc_output, training, look_ahead_mask, padding_mask):
        # x dài 49, enc_output.shape=(150, 128), look_ahead_mask.shape=(49, 49), dec_padding_mask.shape=(1, 150)
        seq_len = tf.shape(x)[1]
        attention_weights = {}
        
        # Lấy embedding từ x
        # create word embeddings 
        x = self.embedding(x) # shape=(49, 128)
        
        # Nhân x với sqrt(emb_dim)
        # scale embeddings by multiplying by the square root of their dimension
        x *= tf.math.sqrt(tf.cast(self.embedding_dim, tf.float32)) # shape=(49, 128)
        
        # Cộng thêm positional encodingds
        # add positional encodings to word embedding
        x += self.pos_encoding[:, :seq_len, :] # shape=(49, 128)

        # apply a dropout layer to x
        # use `training=training`
        x = self.dropout(inputs=x, training=training)

        # use a for loop to pass x through a stack of decoder layers and update attention_weights (~4 lines total)
        for i in range(self.num_layers): # num_layers=2, xem tiếp ở DecoderLayer

            # pass x and the encoder output through a stack of decoder layers and save the attention weights
            # of block 1 and 2 (~1 line)
            x, block1, block2 = self.dec_layers[i](x=x, enc_output=enc_output, training=training, look_ahead_mask=look_ahead_mask, padding_mask=padding_mask)

            #update attention_weights dictionary with the attention weights of block 1 and block 2
            attention_weights['decoder_layer{}_block1_self_att'.format(i+1)] = block1
            attention_weights['decoder_layer{}_block2_decenc_att'.format(i+1)] = block2
        # x.shape=(49, 128)

        return x, attention_weights

### 2.6. Transformer

<img src="images/transformer.png" height=500/>

In [90]:
class Transformer(tf.keras.Model):
    def __init__(self, num_layers, embedding_dim, num_heads, fully_connected_dim, input_vocab_size, 
               target_vocab_size, max_positional_encoding_input,
               max_positional_encoding_target, dropout_rate=0.1, layernorm_eps=1e-6):
        super(Transformer, self).__init__()

        self.encoder = Encoder(num_layers=num_layers,
                               embedding_dim=embedding_dim,
                               num_heads=num_heads,
                               fully_connected_dim=fully_connected_dim,
                               input_vocab_size=input_vocab_size,
                               maximum_position_encoding=max_positional_encoding_input,
                               dropout_rate=dropout_rate,
                               layernorm_eps=layernorm_eps)

        self.decoder = Decoder(num_layers=num_layers, 
                               embedding_dim=embedding_dim,
                               num_heads=num_heads,
                               fully_connected_dim=fully_connected_dim,
                               target_vocab_size=target_vocab_size, 
                               maximum_position_encoding=max_positional_encoding_target,
                               dropout_rate=dropout_rate,
                               layernorm_eps=layernorm_eps)

        self.final_layer = tf.keras.layers.Dense(target_vocab_size, activation='softmax')
    
    def call(self, input_sentence, output_sentence, training, enc_padding_mask, look_ahead_mask, dec_padding_mask):
        # input_sentence dài 150, output_sentence dài 49, enc_padding_mask=dec_padding_mask có shape(1, 150), look_ahead_mask.shape=(49, 49)
        
        # call self.encoder with the appropriate arguments to get the encoder output
        enc_output = self.encoder(x=input_sentence, training=training, mask=enc_padding_mask)
        # enc_output.shape=(150, 128), xem tiếp ở Encoder
        
        # call self.decoder with the appropriate arguments to get the decoder output
        # dec_output.shape == (batch_size, tar_seq_len, fully_connected_dim)
        dec_output, attention_weights = self.decoder(x=output_sentence, enc_output=enc_output, training=training, look_ahead_mask=look_ahead_mask, padding_mask=dec_padding_mask)
        # dec_output.shape=(49, 128)
        
        # Đi qua một FC softmax có units=vocab_size=34250
        # pass decoder output through a linear layer and softmax (~1 line)
        final_output = self.final_layer(dec_output) # shape=(49, 34250)

        return final_output, attention_weights

### 2.7. Training

In [91]:
# Define the model parameters
num_layers = 2
embedding_dim = 128
fully_connected_dim = 128
num_heads = 2
positional_encoding_length = 256

# Initialize the model
transformer = Transformer(
    num_layers, 
    embedding_dim, 
    num_heads, 
    fully_connected_dim,
    vocab_size, 
    vocab_size, 
    positional_encoding_length, 
    positional_encoding_length,
)


class CustomSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, d_model, warmup_steps=4000):
        super(CustomSchedule, self).__init__()
        self.d_model = tf.cast(d_model, dtype=tf.float32)
        self.warmup_steps = warmup_steps
    
    def __call__(self, step):
        step = tf.cast(step, dtype=tf.float32)
        arg1 = tf.math.rsqrt(step)
        arg2 = step * (self.warmup_steps ** -1.5)

        return tf.math.rsqrt(self.d_model) * tf.math.minimum(arg1, arg2)

learning_rate = CustomSchedule(embedding_dim)
optimizer = tf.keras.optimizers.Adam(0.0002, beta_1=0.9, beta_2=0.98, epsilon=1e-9)
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False, reduction='none')


def masked_loss(real, pred):
    mask = tf.math.logical_not(tf.math.equal(real, 0))
    loss_ = loss_object(real, pred)

    mask = tf.cast(mask, dtype=loss_.dtype)
    loss_ *= mask

    return tf.reduce_sum(loss_)/tf.reduce_sum(mask)


train_loss = tf.keras.metrics.Mean(name='train_loss')

@tf.function
def train_step(model, inp, tar):  # Xét một sample, inp là vector dài 150 và tar là vector dài 50
    # e.g.:  inp=[7 722 242 323 516  86 722   8   0  0 ...  0]
    #        tar=[7 722 1563  242  323  4  516  8  0 ... 0] 
    tar_inp = tar[:, :-1] # vector dài 49
    tar_real = tar[:, 1:] # vector dài 49

    # Create masks
    # Hàm create_padding_mask tạo ra vector dài bằng arg có giá trị 1 tại vị trí arg#0
    enc_padding_mask = create_padding_mask(inp) # shape=(1, 150), =[[1  1  1  1  1  1  1  1  0  0 ... 0]]
    
    # Hàm look_ahead_mask tạo ra matrix vuông có shape=arg
    look_ahead_mask = create_look_ahead_mask(tf.shape(tar_inp)[1]) # shape=(49, 49)
    # [[1. 0. 0. ... 0. 0. 0.]
    #  [1. 1. 0. ... 0. 0. 0.]
    #  [1. 1. 1. ... 0. 0. 0.]
    #  ...
    #  [1. 1. 1. ... 1. 1. 0.]
    #  [1. 1. 1. ... 1. 1. 1.]]
    dec_padding_mask = create_padding_mask(inp) # Notice that both encoder and decoder padding masks are equal

    with tf.GradientTape() as tape: # Xem tiếp ở Transformer
        predictions, _ = model(
            input_sentence=inp,
            output_sentence=tar_inp, 
            training=True, 
            enc_padding_mask=enc_padding_mask, 
            look_ahead_mask=look_ahead_mask, 
            dec_padding_mask=dec_padding_mask
        )
        loss = masked_loss(tar_real, predictions)

    gradients = tape.gradient(loss, transformer.trainable_variables)    
    optimizer.apply_gradients(zip(gradients, transformer.trainable_variables))

    train_loss(loss)

In [92]:
def next_word(model, encoder_input, output):
    # Create a padding mask for the input (encoder)
    enc_padding_mask = create_padding_mask(encoder_input)
    # Create a look-ahead mask for the output
    look_ahead_mask = create_look_ahead_mask(tf.shape(output)[1])
    # Create a padding mask for the input (decoder)
    dec_padding_mask = create_padding_mask(encoder_input)

    # Run the prediction of the next word with the transformer model
    predictions, attention_weights = model(
        input_sentence=encoder_input,
        output_sentence=output,
        training=False,
        enc_padding_mask=enc_padding_mask,
        look_ahead_mask=look_ahead_mask,
        dec_padding_mask=dec_padding_mask
    )

    predictions = predictions[: ,-1:, :]
    predicted_id = tf.cast(tf.argmax(predictions, axis=-1), tf.int32)
    
    return predicted_id


def summarize(model, input_document):
    input_document = tokenizer.texts_to_sequences([input_document])
    input_document = tf.keras.preprocessing.sequence.pad_sequences(input_document, maxlen=encoder_maxlen, padding='post', truncating='post')
    encoder_input = tf.expand_dims(input_document[0], 0)
    
    output = tf.expand_dims([tokenizer.word_index["[SOS]"]], 0)
    
    for i in range(decoder_maxlen):
        predicted_id = next_word(model, encoder_input, output)
        output = tf.concat([output, predicted_id], axis=-1)
        
        if predicted_id == tokenizer.word_index["[EOS]"]:
            break

    return tokenizer.sequences_to_texts(output.numpy())[0]  # since there is just one translated document

In [93]:
test_example = 0
true_summary = summary_test[test_example]
true_document = document_test[test_example]

# Define the number of epochs
epochs = 20

# Training loop
for epoch in range(epochs):
    
    start = time.time()
    train_loss.reset_state()
    number_of_batches=len(list(enumerate(dataset)))

    for (batch, (inp, tar)) in enumerate(dataset):
        print(f'Epoch {epoch+1}, Batch {batch+1}/{number_of_batches}', end='\r')
        train_step(transformer, inp, tar)
    
    print (f'Epoch {epoch+1}, Loss {train_loss.result():.4f}')
    
    print (f'Time taken for one epoch: {time.time() - start} sec')
    print('Example summarization on the test set:')
    print('  True summarization:')
    print(f'    {true_summary}')
    print('  Predicted summarization:')
    print(f'    {summarize(transformer, true_document)}\n')

2025-02-27 13:07:38.584102: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


2025-02-27 13:14:01.784750: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch 1, Loss 7.8933
Time taken for one epoch: 383.26252937316895 sec
Example summarization on the test set:
  True summarization:
    [SOS] hannah needs betty's number but amanda doesn't have it. she needs to contact larry. [EOS]
  Predicted summarization:
    [SOS] and [EOS]



2025-02-27 13:14:01.996312: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


2025-02-27 13:19:33.378108: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


...

2025-02-27 14:57:34.052122: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch 19, Loss 3.8738
Time taken for one epoch: 395.780476808548 sec
Example summarization on the test set:
  True summarization:
    [SOS] hannah needs betty's number but amanda doesn't have it. she needs to contact larry. [EOS]
  Predicted summarization:
    [SOS] hannah has just arrived to her house and will be at the park [EOS]



2025-02-27 14:57:35.241629: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


2025-02-27 15:04:02.705283: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch 20, Loss 3.7810
Time taken for one epoch: 387.50602293014526 sec
Example summarization on the test set:
  True summarization:
    [SOS] hannah needs betty's number but amanda doesn't have it. she needs to contact larry. [EOS]
  Predicted summarization:
    [SOS] hannah has just arrived to the wedding on saturday hannah will be at the park [EOS]



## 3. Summarization

In [94]:
training_set_example = 0

# Check a summary of a document from the training set
print('Training set example:')
print(document[training_set_example])
print('\nHuman written summary:')
print(summary[training_set_example])
print('\nModel written summary:')
print(summarize(transformer, document[training_set_example]))

Training set example:
[SOS] amanda: i baked  cookies. do you want some?  jerry: sure!  amanda: i'll bring you tomorrow :-) [EOS]

Human written summary:
[SOS] amanda baked cookies and will bring jerry some tomorrow. [EOS]

Model written summary:
[SOS] amanda wants to bring some cookies tomorrow [EOS]
